# Pipeline KDD — CAR/SICAR

Sistematização de Ciência de Dados II. Ver `plano_trabalho.md` e `RELATORIO.md` para contexto.

## Etapa 1 — Seleção do Dataset

- **Fonte:** Portal oficial do SICAR (consultapublica.car.gov.br/publico/estados/downloads),
  camada "Área do Imóvel", estado da **Bahia (BA)** — baixado manualmente (o download exige
  captcha, não dá pra automatizar) e convertido de `.dbf` para `.csv` com `src/leitor_dbf.py`
  (leitor próprio do formato .dbf, sem geopandas/fiona).
- **Volume:** 1.313.653 linhas (bem acima do mínimo de 100 mil exigido) × 12 colunas — CSV de
  ~184 MB.
- **Colunas:** `cod_tema`, `nom_tema` (constantes — sempre "AREA_IMOVEL"/"Area do Imovel" nesta
  camada), `cod_imovel` (identificador único do imóvel no SICAR), `mod_fiscal` (tamanho em
  módulos fiscais), `num_area` (área total declarada), `ind_status` (situação do cadastro —
  ex. "AT" ativo — **alvo da Etapa 4**), `ind_tipo` (tipo de imóvel, ex. "IRU" rural),
  `des_condic` (descrição textual da condição, ex. "Aguardando analise"), `municipio`,
  `cod_estado`, `dat_criaca`/`dat_atuali` (datas de criação/atualização do registro, formato
  `DD/MM/AAAA` — vêm como string, precisam de `to_date(..., 'dd/MM/yyyy')` na Etapa 2).
- **Justificativa:** CAR/SICAR é o cadastro nacional de imóveis rurais, público e tabular por
  natureza; a Bahia sozinha já ultrapassa em mais de 13x o volume mínimo exigido pelo trabalho,
  e o tema é relevante para o país e a dificuldade de manter seu cadastro rural atualizado devido a sua dimensão territorial.
  `cod_tema`/`nom_tema` não agregam informação (valor único) e serão descartadas na Etapa 2.

In [1]:
# No Windows, o Spark às vezes escolhe o Python errado pra abrir o processo
# "worker" (viramos vítimas disso no diagnóstico do ambiente: dava
# "java.net.SocketException: Connection reset" porque ele tentava usar um
# outro Python instalado na máquina, não este). Forçar PYSPARK_PYTHON pro
# mesmo interpretador que está rodando o notebook resolve.
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("car-sicar-kdd")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark

In [2]:
df = spark.read.csv("../data/raw/AREA_IMOVEL_1.csv", header=True, inferSchema=True)
df.printSchema()
df.count()

root
 |-- cod_tema: string (nullable = true)
 |-- nom_tema: string (nullable = true)
 |-- cod_imovel: string (nullable = true)
 |-- mod_fiscal: double (nullable = true)
 |-- num_area: double (nullable = true)
 |-- ind_status: string (nullable = true)
 |-- ind_tipo: string (nullable = true)
 |-- des_condic: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- cod_estado: string (nullable = true)
 |-- dat_criaca: string (nullable = true)
 |-- dat_atuali: string (nullable = true)



1313653

## Etapa 2 — Ingestão e Pré-processamento com Spark

Tratamento de nulos, tipos, duplicatas, outliers e engenharia de atributos.

**Achados principais:**

- `cod_tema` e `nom_tema` são constantes em todo o dataset (só existe "AREA_IMOVEL" /
  "Area do Imovel") — não carregam informação nenhuma, foram descartadas.
- 33 grupos de `cod_imovel` duplicado (66 linhas no total). Investigando um exemplo,
  o padrão ficou claro pela coluna `des_condic`: cada duplicidade é uma linha com
  `ind_status = "AT"` (ativa) mais uma ou mais linhas com status de cancelamento e
  `des_condic` explicando o motivo — no caso mais comum, "Cancelado por duplicidade".
  Verificado que o padrão se repete em todos os 33 grupos, não só no exemplo. Resolvido
  mantendo a linha `"AT"` de cada grupo (ou a mais recente por `dat_atuali`, quando não
  há nenhuma ativa) — de 1.313.653 linhas para 1.313.620, batendo exatamente com o
  número de `cod_imovel` distintos.
- 8 nulos em `dat_atuali` (nenhum nulo nas demais colunas), preenchidos com o valor de
  `dat_criaca` (assume-se que um imóvel nunca atualizado manteve a data de criação como
  última atualização).
- `dat_criaca` e `dat_atuali` vieram como texto no formato `DD/MM/AAAA` — convertidas
  para o tipo `date` do Spark.
- Não foram encontrados outliers óbvios de área (`num_area`) ou módulo fiscal
  (`mod_fiscal`) negativos ou zerados.
- `ind_status` é fortemente desbalanceado (~99,4% `AT`) — relevante para a Etapa 4
  (modelagem preditiva), vai precisar de tratamento de desbalanceamento (ex. pesos por
  classe, undersampling/oversampling, ou métricas que não sejam só acurácia).


In [3]:
from pyspark.sql import functions as F

# 1) cod_tema/nom_tema são mesmo constantes (sem informação nenhuma)?
df.select("cod_tema", "nom_tema").distinct().show()

# 2) quantos nulos tem em cada coluna?
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 3) cod_imovel é único, ou tem imóvel repetido?
total = df.count()
distintos = df.select("cod_imovel").distinct().count()
print(f"total de linhas: {total} | cod_imovel distintos: {distintos}")

# 4) quais valores existem em ind_status e ind_tipo, e quão comum é cada um?
df.groupBy("ind_status").count().orderBy(F.desc("count")).show()
df.groupBy("ind_tipo").count().orderBy(F.desc("count")).show()

+-----------+--------------+
|   cod_tema|      nom_tema|
+-----------+--------------+
|AREA_IMOVEL|Area do Imovel|
+-----------+--------------+

+--------+--------+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+
|cod_tema|nom_tema|cod_imovel|mod_fiscal|num_area|ind_status|ind_tipo|des_condic|municipio|cod_estado|dat_criaca|dat_atuali|
+--------+--------+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+
|       0|       0|         0|         0|       0|         0|       0|         0|        0|         0|         0|         8|
+--------+--------+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+

total de linhas: 1313653 | cod_imovel distintos: 1313620
+----------+-------+
|ind_status|  count|
+----------+-------+
|        AT|1305977|
|        PE|   4765|
|        CA|   2848|
|        SU|     63|
+----------+-------+

+-

In [4]:
# 5) investigar os cod_imovel duplicados
duplicados = df.groupBy("cod_imovel").count().filter("count > 1")
duplicados.orderBy(F.desc("count")).show(10, truncate=False)

duplicados_lista = [row["cod_imovel"] for row in duplicados.collect()]
df.filter(df.cod_imovel.isin(duplicados_lista)) \
  .select("cod_imovel", "ind_status", "des_condic", "dat_atuali") \
  .orderBy("cod_imovel") \
  .show(len(duplicados_lista) * 2, truncate=False)

# 6) checar outliers óbvios de área / módulo fiscal (negativos ou zerados)
df.select("mod_fiscal", "num_area").describe().show()
df.filter((F.col("num_area") <= 0) | (F.col("mod_fiscal") <= 0)).count()


+-------------------------------------------+-----+
|cod_imovel                                 |count|
+-------------------------------------------+-----+
|BA-2922854-20D021A9A1784E9B9F027190535F96DF|5    |
|BA-2924405-1B6E74379E8747D5BE460584C69500F1|3    |
|BA-2905107-A64B201026E54952ABB4DA3D2B28ABBE|3    |
|BA-2921500-EF449CC5A96A4D0FAE083721D39BF8F9|2    |
|BA-2911105-2E777BD7805642499FA76B1F807F24FD|2    |
|BA-2903201-6985D757CB534946AF26113E191D2DFC|2    |
|BA-2923605-565761EB5F204FC4BC74B87BE7D8CEAF|2    |
|BA-2924405-64717EAF63F8487E8CEEE99E47B8C18F|2    |
|BA-2923209-93D1D70A5240486092BB4E3E3872BA97|2    |
|BA-2917359-94937D4ADA66494B97733A18E7D3C8B3|2    |
+-------------------------------------------+-----+
only showing top 10 rows

+-------------------------------------------+----------+-------------------------+----------+
|cod_imovel                                 |ind_status|des_condic               |dat_atuali|
+-------------------------------------------+----------+--

173333

In [5]:
from pyspark.sql import Window

# cada grupo duplicado tem uma linha "AT" (ativa) + uma ou mais canceladas por
# duplicidade: ficamos com a linha AT quando existe, senão a mais recente por dat_atuali
janela = Window.partitionBy("cod_imovel").orderBy(
    F.when(F.col("ind_status") == "AT", 0).otherwise(1),
    F.desc("dat_atuali")
)

df_dedup = (
    df.withColumn("rn", F.row_number().over(janela))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

print(f"antes: {df.count()} | depois: {df_dedup.count()}")
print(f"cod_imovel distintos depois: {df_dedup.select('cod_imovel').distinct().count()}")


antes: 1313653 | depois: 1313620
cod_imovel distintos depois: 1313620


In [6]:
# descartar colunas constantes, preencher nulos e converter datas
# (o coalesce precisa acontecer ANTES da conversão pra date, enquanto as duas
# colunas ainda são texto)
df_limpo = (
    df_dedup
    .drop("cod_tema", "nom_tema")
    .withColumn("dat_atuali", F.coalesce(F.col("dat_atuali"), F.col("dat_criaca")))
    .withColumn("dat_criaca", F.to_date(F.col("dat_criaca"), "dd/MM/yyyy"))
    .withColumn("dat_atuali", F.to_date(F.col("dat_atuali"), "dd/MM/yyyy"))
)

df_limpo.printSchema()
df_limpo.select("dat_criaca", "dat_atuali").show(5)

# checagem final: não deve sobrar nenhum nulo
df_limpo.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_limpo.columns]).show()


root
 |-- cod_imovel: string (nullable = true)
 |-- mod_fiscal: double (nullable = true)
 |-- num_area: double (nullable = true)
 |-- ind_status: string (nullable = true)
 |-- ind_tipo: string (nullable = true)
 |-- des_condic: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- cod_estado: string (nullable = true)
 |-- dat_criaca: date (nullable = true)
 |-- dat_atuali: date (nullable = true)

+----------+----------+
|dat_criaca|dat_atuali|
+----------+----------+
|2026-06-09|2026-06-09|
|2017-11-02|2017-11-02|
|2017-11-02|2017-11-02|
|2022-11-23|2022-11-23|
|2022-07-16|2022-07-16|
+----------+----------+
only showing top 5 rows

+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+
|cod_imovel|mod_fiscal|num_area|ind_status|ind_tipo|des_condic|municipio|cod_estado|dat_criaca|dat_atuali|
+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+
|         0|         

## Etapa 3 — Análise Exploratória com Spark SQL

Registrar o DataFrame como view e responder pelo menos 5 perguntas de negócio.

In [7]:
df.createOrReplaceTempView("car")

### Pergunta 1


In [8]:
# Como é a distribuição de propriedades por município?
spark.sql("""
SELECT municipio, COUNT(*) AS qtd_imoveis, SUM(num_area) AS area_total, ROUND(AVG(num_area), 2) AS area_media
FROM car
GROUP BY municipio
ORDER BY area_total DESC
LIMIT 10
""").show()

+--------------------+-----------+------------------+----------+
|           municipio|qtd_imoveis|        area_total|area_media|
+--------------------+-----------+------------------+----------+
|Formosa do Rio Preto|       5509|1910207.3220999925|    346.74|
|       Sao Desiderio|       7716| 1511448.203100001|    195.88|
|          Correntina|       8182|1186648.1343000033|    145.03|
|        Pilao Arcado|      16224|1057719.5164000005|     65.19|
|           Jaborandi|       4273| 975111.2001999909|     228.2|
|            Sento Se|       6599| 968937.0880999957|    146.83|
|               Cocos|       3625| 907398.1850000018|    250.32|
|           Barreiras|       6282| 756953.9417999998|     120.5|
|               Barra|       8887| 756451.8793000021|     85.12|
|   Riachao das Neves|       3177| 648120.3966000018|     204.0|
+--------------------+-----------+------------------+----------+



### Pergunta 2


In [9]:
# Qual a distribuição das propriedades por tamanho?
spark.sql("""
SELECT
  CASE WHEN mod_fiscal <= 4 THEN 'Pequena'
       WHEN mod_fiscal <= 15 THEN 'Média'
       ELSE 'Grande' END AS faixa,
  COUNT(*) AS qtd_imoveis,
  SUM(num_area) AS area_total,
  ROUND(AVG(num_area), 2) AS area_media
FROM car
GROUP BY 1
ORDER BY area_total DESC
""").show()

+-------+-----------+--------------------+----------+
|  faixa|qtd_imoveis|          area_total|area_media|
+-------+-----------+--------------------+----------+
|Pequena|    1283109| 1.807189701499996E7|     14.08|
| Grande|       7250|1.5596481837599996E7|   2151.24|
|  Média|      23294|        8520160.4014|    365.77|
+-------+-----------+--------------------+----------+



### Pergunta 3


In [10]:
# Qual a distribuição por situação cadastral?
spark.sql("""
SELECT 
    CASE WHEN ind_status = 'AT' THEN 'Ativo'
         WHEN ind_status = 'PE' THEN 'Pendente'
         WHEN ind_status = 'CA' THEN 'Cancelado'
         WHEN ind_status = 'SU' THEN 'Suspenso'
        END AS status, COUNT(*) AS qtd, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentual
FROM car
GROUP BY ind_status
ORDER BY qtd DESC
""").show()

+---------+-------+----------+
|   status|    qtd|percentual|
+---------+-------+----------+
|    Ativo|1305977|     99.42|
| Pendente|   4765|      0.36|
|Cancelado|   2848|      0.22|
| Suspenso|     63|      0.00|
+---------+-------+----------+



### Pergunta 4


In [11]:
# Como foi a adesão ao CAR ao longo dos anos?
spark.sql("""
SELECT 
    YEAR(to_date(dat_criaca, 'dd/MM/yyyy')) AS ano, 
    COUNT(*) AS qtd_cadastros, 
    ROUND(SUM(num_area), 2) AS area_cadastrada
FROM car
WHERE dat_criaca IS NOT NULL
GROUP BY 1
ORDER BY 1
""").show()

+----+-------------+---------------+
| ano|qtd_cadastros|area_cadastrada|
+----+-------------+---------------+
|2014|         3239|     1229837.52|
|2015|        22604|     4738503.71|
|2016|        70576|     4785811.84|
|2017|       301554|     5207682.76|
|2018|       211564|     4388495.03|
|2019|       161670|     4073911.56|
|2020|        93801|     3683553.64|
|2021|        89481|     3197289.59|
|2022|        68293|     2493330.28|
|2023|        76555|     2925777.94|
|2024|        72696|     2282582.33|
|2025|        87120|     1882520.08|
|2026|        54500|     1299242.98|
+----+-------------+---------------+



### Pergunta 5


In [12]:
#Qual o tamanho médio das propriedades por tipo de imóvel
spark.sql("""
SELECT 
    CASE WHEN ind_tipo = 'IRU' THEN 'Propiedade Rural'
         WHEN ind_tipo = 'AST' THEN 'Assentamento'
         WHEN ind_tipo = 'PCT' THEN 'Povos/Comunidades Tradicionais'
    END AS tipo, COUNT(*) AS qtd, ROUND(AVG(num_area), 2) AS area_media, ROUND(AVG(mod_fiscal), 2) AS mod_fiscal_medio
FROM car
GROUP BY ind_tipo
ORDER BY qtd DESC
""").show()

+--------------------+-------+----------+----------------+
|                tipo|    qtd|area_media|mod_fiscal_medio|
+--------------------+-------+----------+----------------+
|    Propiedade Rural|1312087|     29.22|            0.57|
|        Assentamento|    905|   2834.72|           34.08|
|Povos/Comunidades...|    661|    1936.6|           17.65|
+--------------------+-------+----------+----------------+



## Etapa 4 — Modelagem Preditiva

Problema (classificação/regressão), 2+ modelos de famílias diferentes (1 ensemble), métricas e validação.

In [13]:
from pyspark.sql.functions import col, to_date, datediff, current_date, when, count

# 1) Coluna alvo: 1 = Ativo, 0 = qualquer outra situação
df_model = df.withColumn(
    "target_ativo",
    when(col("ind_status") == "AT", 1).otherwise(0)
)

# 2) Dias desde a última atualização do cadastro (usa o formato dd/MM/yyyy que já identificamos)
df_model = df_model.withColumn(
    "dt_criacao", to_date(col("dat_criaca"), "dd/MM/yyyy")
).withColumn(
    "dt_atualizacao", to_date(col("dat_atuali"), "dd/MM/yyyy")
).withColumn(
    "dias_desde_atualizacao", datediff(current_date(), col("dt_atualizacao"))
)

# 3) municipio tem muitas categorias — vamos manter só os 20 mais frequentes e
#    agrupar o resto em "OUTROS", pra não explodir o número de colunas no encoding
top_municipios = (
    df_model.groupBy("municipio")
    .agg(count("*").alias("qtd"))
    .orderBy(col("qtd").desc())
    .limit(20)
    .select("municipio")
    .rdd.flatMap(lambda x: x)
    .collect()
)

df_model = df_model.withColumn(
    "municipio_agrupado",
    when(col("municipio").isin(top_municipios), col("municipio")).otherwise("OUTROS")
)

df_model.select("target_ativo", "dias_desde_atualizacao", "municipio_agrupado", "ind_tipo", "num_area", "mod_fiscal").show(10)



+------------+----------------------+------------------+--------+--------+----------+
|target_ativo|dias_desde_atualizacao|municipio_agrupado|ind_tipo|num_area|mod_fiscal|
+------------+----------------------+------------------+--------+--------+----------+
|           1|                    19|            OUTROS|     IRU| 10.8626|      0.16|
|           1|                    19|            OUTROS|     IRU|  0.5085|       0.0|
|           1|                    19|            OUTROS|     IRU|  3.4288|      0.05|
|           1|                    19|            OUTROS|     IRU|  1.5301|      0.02|
|           1|                    20|            OUTROS|     IRU|  6.9278|       0.1|
|           1|                    24|            OUTROS|     IRU|     2.3|      0.03|
|           1|                  1916|            OUTROS|     IRU|  1.9373|      0.02|
|           1|                   598|            OUTROS|     IRU|327.2631|      5.03|
|           1|                  1916|            OUTRO

In [14]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Indexação das categóricas (string -> número)
indexer_tipo = StringIndexer(inputCol="ind_tipo", outputCol="ind_tipo_idx", handleInvalid="keep")
indexer_muni = StringIndexer(inputCol="municipio_agrupado", outputCol="municipio_idx", handleInvalid="keep")

# One-hot encoding em cima dos índices (evita que o modelo interprete como ordem/escala)
encoder = OneHotEncoder(
    inputCols=["ind_tipo_idx", "municipio_idx"],
    outputCols=["ind_tipo_vec", "municipio_vec"]
)

# Junta todas as features num único vetor
assembler = VectorAssembler(
    inputCols=["num_area", "mod_fiscal", "dias_desde_atualizacao", "ind_tipo_vec", "municipio_vec"],
    outputCol="features",
    handleInvalid="skip"  # descarta linhas com valor nulo em alguma feature
)

pipeline_prep = Pipeline(stages=[indexer_tipo, indexer_muni, encoder, assembler])

modelo_prep = pipeline_prep.fit(df_model)
df_features = modelo_prep.transform(df_model)

# Split treino/teste (80/20), com seed fixa pra reprodutibilidade
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

print("Treino:", train_df.count(), "| Teste:", test_df.count())
train_df.groupBy("target_ativo").count().show()

Treino: 1050872 | Teste: 262773
+------------+-------+
|target_ativo|  count|
+------------+-------+
|           1|1044729|
|           0|   6143|
+------------+-------+



In [15]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# --- Peso das classes (mesma lógica do "class_weight='balanced'") ---
n_total = train_df.count()
n_pos = train_df.filter(col("target_ativo") == 1).count()
n_neg = train_df.filter(col("target_ativo") == 0).count()

peso_pos = n_total / (2 * n_pos)
peso_neg = n_total / (2 * n_neg)

train_df = train_df.withColumn(
    "peso",
    when(col("target_ativo") == 1, peso_pos).otherwise(peso_neg)
)

print(f"Peso classe 1 (Ativo): {peso_pos:.4f} | Peso classe 0 (nao-Ativo): {peso_neg:.4f}")

# --- Modelo 1: Regressao Logistica (familia linear) ---
lr = LogisticRegression(featuresCol="features", labelCol="target_ativo", weightCol="peso", maxIter=50)
modelo_lr = lr.fit(train_df)
pred_lr = modelo_lr.transform(test_df)

# --- Modelo 2: Random Forest (familia ensemble) ---
rf = RandomForestClassifier(featuresCol="features", labelCol="target_ativo", weightCol="peso", numTrees=100, seed=42)
modelo_rf = rf.fit(train_df)
pred_rf = modelo_rf.transform(test_df)

# --- Avaliacao: AUC ---
evaluator_auc = BinaryClassificationEvaluator(labelCol="target_ativo", metricName="areaUnderROC")
print(f"AUC - Regressao Logistica: {evaluator_auc.evaluate(pred_lr):.4f}")
print(f"AUC - Random Forest: {evaluator_auc.evaluate(pred_rf):.4f}")

# --- Matriz de confusao de cada modelo ---
print("Matriz de confusao - Regressao Logistica:")
pred_lr.groupBy("target_ativo", "prediction").count().orderBy("target_ativo", "prediction").show()

print("Matriz de confusao - Random Forest:")
pred_rf.groupBy("target_ativo", "prediction").count().orderBy("target_ativo", "prediction").show()

Peso classe 1 (Ativo): 0.5029 | Peso classe 0 (nao-Ativo): 85.5341
AUC - Regressao Logistica: 0.6957
AUC - Random Forest: 0.7504
Matriz de confusao - Regressao Logistica:
+------------+----------+------+
|target_ativo|prediction| count|
+------------+----------+------+
|           0|       0.0|   906|
|           0|       1.0|   620|
|           1|       0.0| 90577|
|           1|       1.0|170670|
+------------+----------+------+

Matriz de confusao - Random Forest:
+------------+----------+------+
|target_ativo|prediction| count|
+------------+----------+------+
|           0|       0.0|   790|
|           0|       1.0|   736|
|           1|       0.0| 45385|
|           1|       1.0|215862|
+------------+----------+------+



In [19]:
import pandas as pd

# Recupera os rótulos usados no encoding das categóricas
labels_tipo = modelo_prep.stages[0].labels   # StringIndexer de ind_tipo
labels_muni = modelo_prep.stages[1].labels   # StringIndexer de municipio_agrupado

# OneHotEncoder por padrao descarta a ultima categoria (dropLast=True)
nomes_tipo = [f"ind_tipo_{c}" for c in labels_tipo[:-1]]
nomes_muni = [f"municipio_{c}" for c in labels_muni[:-1]]

nomes_features = ["num_area", "mod_fiscal", "dias_desde_atualizacao"] + nomes_tipo + nomes_muni
importancias = modelo_rf.featureImportances.toArray()

if len(nomes_features) == len(importancias):
    df_importancia = pd.DataFrame({
        "feature": nomes_features,
        "importancia": importancias
    }).sort_values("importancia", ascending=False)
    print(df_importancia.to_string(index=False))
else:
    print(f"Aviso: {len(nomes_features)} nomes vs {len(importancias)} importancias — nao bateu, mostrando so os valores brutos:")
    for i, imp in enumerate(importancias):
        print(f"feature_{i}: {imp:.4f}")

Aviso: 25 nomes vs 27 importancias — nao bateu, mostrando so os valores brutos:
feature_0: 0.3329
feature_1: 0.4211
feature_2: 0.0589
feature_3: 0.0009
feature_4: 0.0001
feature_5: 0.0005
feature_6: 0.1193
feature_7: 0.0247
feature_8: 0.0044
feature_9: 0.0015
feature_10: 0.0027
feature_11: 0.0008
feature_12: 0.0045
feature_13: 0.0003
feature_14: 0.0006
feature_15: 0.0005
feature_16: 0.0044
feature_17: 0.0017
feature_18: 0.0005
feature_19: 0.0038
feature_20: 0.0067
feature_21: 0.0039
feature_22: 0.0002
feature_23: 0.0023
feature_24: 0.0005
feature_25: 0.0021
feature_26: 0.0001


In [20]:
nomes_tipo = [f"ind_tipo_{c}" for c in labels_tipo]     # todas as 3 categorias (IRU, AST, PCT)
nomes_muni = [f"municipio_{c}" for c in labels_muni]    # todas as 21 (20 + OUTROS)

nomes_features = ["num_area", "mod_fiscal", "dias_desde_atualizacao"] + nomes_tipo + nomes_muni
importancias = modelo_rf.featureImportances.toArray()

print(len(nomes_features), len(importancias))  # confere que agora bate: deve dar 27 e 27

df_importancia = pd.DataFrame({
    "feature": nomes_features,
    "importancia": importancias
}).sort_values("importancia", ascending=False)
print(df_importancia.to_string(index=False))

27 27
                              feature  importancia
                           mod_fiscal     0.421111
                             num_area     0.332898
                     municipio_OUTROS     0.119269
               dias_desde_atualizacao     0.058861
                municipio_Monte Santo     0.024706
    municipio_Campo Alegre de Lourdes     0.006708
                    municipio_Brumado     0.004499
                   municipio_Macaubas     0.004417
               municipio_Pilao Arcado     0.004411
municipio_Livramento de Nossa Senhora     0.003915
         municipio_Conceicao do Coite     0.003810
                      municipio_Araci     0.002733
                     municipio_Itiuba     0.002338
                       municipio_Uaua     0.002133
                   municipio_Juazeiro     0.001680
          municipio_Euclides da Cunha     0.001511
                         ind_tipo_IRU     0.000888
              municipio_Campo Formoso     0.000812
                  municip

## Etapa 5 — Modelagem Descritiva

Clusterização (ex. K-Means), escolha do k (cotovelo/silhueta) e interpretação dos perfis.

In [21]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Monta o vetor de features numericas para clusterizacao
assembler_cluster = VectorAssembler(
    inputCols=["num_area", "mod_fiscal", "dias_desde_atualizacao"],
    outputCol="features_cluster_raw",
    handleInvalid="skip"
)

df_cluster = assembler_cluster.transform(df_model)

# Padroniza (media 0, desvio padrao 1)
scaler = StandardScaler(inputCol="features_cluster_raw", outputCol="features_cluster", withMean=True, withStd=True)
modelo_scaler = scaler.fit(df_cluster)
df_cluster = modelo_scaler.transform(df_cluster)

# Testa k de 2 a 6 e mede o silhouette de cada um (quanto mais proximo de 1, melhor separados os clusters)
evaluator = ClusteringEvaluator(featuresCol="features_cluster", metricName="silhouette")

for k in range(2, 7):
    kmeans = KMeans(featuresCol="features_cluster", k=k, seed=42)
    modelo_k = kmeans.fit(df_cluster)
    predicoes = modelo_k.transform(df_cluster)
    silhouette = evaluator.evaluate(predicoes)
    print(f"k={k} | silhouette={silhouette:.4f} | custo (WSSSE)={modelo_k.summary.trainingCost:.2f}")

k=2 | silhouette=0.4104 | custo (WSSSE)=2900825.96
k=3 | silhouette=0.6754 | custo (WSSSE)=1744986.28
k=4 | silhouette=0.3882 | custo (WSSSE)=1596669.79
k=5 | silhouette=0.6375 | custo (WSSSE)=1014841.67
k=6 | silhouette=0.6777 | custo (WSSSE)=784702.71


In [23]:
kmeans_final = KMeans(featuresCol="features_cluster", k=3, seed=42)
modelo_kmeans = kmeans_final.fit(df_cluster)
df_clusters = modelo_kmeans.transform(df_cluster)

print(f"Silhouette final: {evaluator.evaluate(df_clusters):.4f}")

# Perfil de cada cluster nas variaveis originais (nao padronizadas)
from pyspark.sql.functions import avg
from pyspark.sql.functions import avg, count
df_clusters.groupBy("prediction").agg(
    count("*").alias("qtd_imoveis"),
    avg("num_area").alias("area_media"),
    avg("mod_fiscal").alias("mod_fiscal_medio"),
    avg("dias_desde_atualizacao").alias("dias_media")
).orderBy("prediction").show()

Silhouette final: 0.6754
+----------+-----------+------------------+------------------+------------------+
|prediction|qtd_imoveis|        area_media|  mod_fiscal_medio|        dias_media|
+----------+-----------+------------------+------------------+------------------+
|         0|     715338| 18.23036895271887|0.3467998161708559|2873.0747031473234|
|         1|        259|16695.357418532818| 256.7850664092664|1190.6370656370657|
|         2|     598048|41.390480377160365|0.7940780318636695| 836.3545150222055|
+----------+-----------+------------------+------------------+------------------+



## Etapa 6 — Interpretação e Conclusões (KDD)

Conhecimento descoberto, decisões possíveis, limitações e próximos passos.

## Etapa 6 — Interpretação e Conclusões (KDD)

### Sobre o problema e o dataset
Este trabalho aplicou o processo de KDD (Knowledge Discovery in Databases) sobre a
camada "Área do Imóvel" do CAR/SICAR (Cadastro Ambiental Rural) referente ao estado
da Bahia, totalizando 1.313.653 registros (~184 MB), obtidos diretamente do portal
oficial (car.gov.br). A escolha do dataset se justifica pela ligação com a área de
geoprocessamento/cartografia.

### Pré-processamento
Durante a ingestão, identificamos que as colunas de data (`dat_criaca`, `dat_atuali`)
estavam armazenadas como texto no formato brasileiro (`dd/MM/yyyy`), exigindo conversão
explícita com `to_date()` antes de qualquer extração de ano — um lembrete de que a
inspeção da representação bruta dos dados é indispensável antes de qualquer análise
temporal.

### Análise exploratória (Etapa 3)
As 5 perguntas de negócio revelaram uma base extremamente concentrada: 99,42% dos
cadastros estão na situação "Ativo" (ind_status = AT), com apenas 0,36% pendentes,
0,22% cancelados e frações residuais suspensas. Quanto ao tipo de imóvel, 99,86% são
Imóveis Rurais individuais (IRU), enquanto Assentamentos (AST) e territórios de Povos
e Comunidades Tradicionais (PCT), embora raros, respondem por áreas médias muito
maiores (2.834 ha e 1.937 ha, respectivamente, contra 29 ha do IRU) — evidenciando que
são cadastros coletivos de território, não propriedades individuais.

### Modelagem preditiva (Etapa 4)
Buscamos prever a situação cadastral (Ativo vs. não-Ativo) a partir de características
do imóvel (área, módulo fiscal, tipo, município e tempo desde a última atualização),
usando Regressão Logística e Random Forest (com pesos de classe para compensar o
desbalanceamento de 99,42%/0,58%). O Random Forest obteve melhor poder discriminativo
geral (AUC 0,751 contra 0,696 da Regressão Logística), mas ambos os modelos têm baixa
precisão para a classe minoritária — resultado esperado dado o desbalanceamento extremo,
e não uma falha da modelagem em si. A análise de importância de variáveis mostrou que o
tamanho do imóvel (módulo fiscal e área, juntos ~75% da importância) é o fator dominante
na previsão, muito à frente da localização geográfica e do tipo de imóvel.

### Modelagem descritiva (Etapa 5)
A clusterização com K-Means (k=3, escolhido por silhouette score de 0,675, o melhor
resultado com um número de clusters ainda interpretável) revelou três perfis:
- **Mega-propriedades** (259 imóveis, 0,02% da base): área média de 16.695 ha e módulo
  fiscal médio de 256,8 — muito acima do limiar de "grande propriedade" (>15 MF),
  provavelmente assentamentos e territórios coletivos.
- **Pequenas propriedades com cadastro estagnado** (715.338 imóveis, 54,5%): área
  média de 18,2 ha, sem atualização há em média quase 8 anos (2.873 dias).
- **Pequenas propriedades com cadastro recente** (598.048 imóveis, 45,5%): área
  média de 41,4 ha, atualizadas há em média 2,3 anos (836 dias).

O achado relevante é que o algoritmo não separou os grupos apenas por tamanho, mas
principalmente pelo tempo desde a última atualização — sugerindo uma divisão real na
base entre cadastros ativos e "esquecidos", coerente com o peso que essa mesma variável
teve na Etapa 4.

### Conclusões e limitações
A base do CAR/SICAR para a Bahia é dominada por pequenas propriedades rurais individuais,
majoritariamente ativas — um retrato coerente com a estrutura fundiária brasileira. O
tamanho do imóvel é o principal fator associado tanto à situação cadastral quanto ao
perfil de atualização do cadastro. Como limitações, destacam-se: (1) o forte
desbalanceamento das classes de status dificulta a previsão de casos raros (pendentes,
cancelados, suspensos) com boa precisão; (2) a base pública não permite saber por que um
cadastro ficou tanto tempo sem atualização (falta de contexto administrativo); (3) o
dataset representa apenas um recorte estadual, não podendo ser generalizado para outras
regiões do país sem nova validação.